# D-04. 임베딩 모델 비교 — MLP vs FCN vs FCN+center loss

- 질문: center loss로 같은 재질 임베딩을 모으면, 프로토타입 거리가 진동 세기 대신 **새 재질**을 잡는가?
- 입력 (모두 147 split, 7626구간 데이터셋 순서)
  - `mlp`: B의 특성 MLP (D-01~03 기준선)
  - `fcn`: C의 FCN, 교차 엔트로피만
  - `fcn-center`: C의 FCN + center loss
- 모델마다 D-01(기존 5개 방법), D-02(원인 진단), D-03(진동과 떨어진 점수 4개)을 한 번에 실행하고 나란히 비교
- 주의: 두 임베딩 모델 모두 τ용 검증 녹화까지 학습에 썼다 → 알려진 재질 통과율이 0.95보다 낮게 나올 수 있다 (center loss는 더)

실행: Colab, CPU로 충분 (모델 3개 × 147 split, 10~20분)

## 0. 환경 준비

In [ ]:
import os
import sys
from pathlib import Path

BRANCH = "main"   # PR 브랜치에서 시험할 때만 그 브랜치 이름으로

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/ML-imu").exists():
        !git clone -q https://github.com/ekgP525/ML-imu.git /content/ML-imu
    os.chdir("/content/ML-imu")
    !git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
    !pip install -q -r requirements.txt
    !python data/download.py

ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path += [str(ROOT / "src"), str(ROOT / "notebooks" / "D_ood")]
print(ROOT)

In [ ]:
# 임베딩 찾기: 내 드라이브의 폴더 → 없으면 MyDrive/ 맨 위의 zip을 Colab 안에 푼다
# 공유받은 폴더는 '공유 문서함' → 오른쪽 클릭 → '바로가기 추가' → 내 드라이브 에 넣어야 보인다
MODELS = ["mlp", "fcn", "fcn-center"]
ZIP_PATTERNS = ["ML-imu_embeddings*.zip", "*embeddings*.zip", "*fcn*.zip"]

if IN_COLAB:
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    OUT = MYDRIVE / "ML-imu_D"
    SEARCH = [MYDRIVE / "ML-imu_embeddings", MYDRIVE / "ML-imu" / "embeddings", MYDRIVE / "embeddings"]
else:
    MYDRIVE = None
    OUT = ROOT / "results"
    SEARCH = []
LOCAL = ROOT / "embeddings" / "all"
SEARCH += [LOCAL, ROOT / "embeddings" / "mlp"]
OUT.mkdir(parents=True, exist_ok=True)


def find_dir(model):
    for d in SEARCH:
        if (d / f"emb_{model}_carpet_0.npy").exists():
            return d
    return None


missing = [m for m in MODELS if find_dir(m) is None]
if missing and MYDRIVE is not None:
    zips = sorted({z for p in ZIP_PATTERNS for z in MYDRIVE.glob(p)})
    LOCAL.mkdir(parents=True, exist_ok=True)
    for zp in zips:
        with zipfile.ZipFile(zp) as zf:
            for info in zf.infolist():
                name = Path(info.filename).name
                if any(name.startswith(f"{k}_{m}_") for k in ("emb", "logits") for m in missing):
                    (LOCAL / name).write_bytes(zf.read(info))
        print("압축 해제:", zp.name)

EMB = {m: find_dir(m) for m in MODELS}
for m, d in EMB.items():
    n = len(list(d.glob(f"emb_{m}_*.npy"))) if d else 0
    print(f"{m:<11} {n:3d}개 임베딩  ←  {d}")
assert all(EMB.values()), "임베딩을 못 찾은 모델이 있어요. 위 경로를 확인하세요."

## 1. 데이터

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recordings import load_dataset, CHANNEL_NAMES
from splits import load_material_folds
from evaluate import log_result
from ood import run_heldout, diagnose_fold, EMBEDDING_SCORERS, SCALE_FREE_SCORERS, DIFFICULTY

X, y, rec = load_dataset("data")
folds = load_material_folds()
vib = X[:, :, CHANNEL_NAMES.index("acc_z")].std(axis=1)

QUIET = ["carpet", "soft_pvc", "soft_tiles", "wood"]
order = QUIET + ["fine_concrete", "concrete", "hard_tiles", "tiled"]
METHODS = ["proto", "scaled", "scaled_nn", "cmaha", "ratio", "maha", "knn", "msp", "energy"]
SCORERS = {**EMBEDDING_SCORERS, **SCALE_FREE_SCORERS}
TAU_METHODS = ("proto", "scaled", "scaled_nn", "cmaha", "ratio")

for m in MODELS:
    for s in folds:
        tag = f"{m}_{s['held_out']}_{s['fold']}"
        assert (EMB[m] / f"emb_{tag}.npy").exists() and (EMB[m] / f"logits_{tag}.npy").exists(), tag
print(X.shape, len(folds), "개 split, 모델", MODELS, "모두 147개 확인")

## 2. 모델별 실행 (방법 9개 + 원인 진단)

In [ ]:
def loader(model):
    def load(held_out, fold):
        tag = f"{model}_{held_out}_{fold}"
        return np.load(EMB[model] / f"emb_{tag}.npy"), np.load(EMB[model] / f"logits_{tag}.npy")
    return load

runs, diags = [], []
for m in MODELS:
    load = loader(m)
    z0, l0 = load(folds[0]["held_out"], folds[0]["fold"])
    assert len(z0) == len(y), (m, z0.shape)
    print(f"{m}: emb {z0.shape}, logits {l0.shape}")
    df_m = run_heldout(load, y, rec, model=m, folds=folds,
                       scorers=SCORERS, tau_methods=TAU_METHODS, vib=vib)
    df_m.to_csv(OUT / f"D_compare_{m}_folds.csv", index=False)
    runs.append(df_m)
    for s in folds:
        diags.append({"model": m, "heldout": s["held_out"], "fold": s["fold"],
                      **diagnose_fold(load(s["held_out"], s["fold"])[0], y, rec, s)})

df = pd.concat(runs, ignore_index=True)
df["group"] = np.where(df.heldout.isin(QUIET), "quiet", "loud")
diag = pd.DataFrame(diags)
diag.to_csv(OUT / "D_compare_diagnosis.csv", index=False)
print(len(df), "행")

## 3. 모델 × 방법 비교

In [ ]:
def model_table(value):
    return df.pivot_table(index="method", columns="model", values=value).loc[METHODS, MODELS]

print("AUROC 평균 (147 split)")
display(model_table("auroc").round(3))
print("FPR@95%TPR 평균 (작을수록 좋음)")
display(model_table("fpr95").round(3))
print("점수와 진동 세기의 상관 (0에 가까울수록 진동과 무관)")
display(model_table("rho_vib").round(3))

g = df.pivot_table(index="method", columns=["model", "group"], values="auroc").loc[METHODS]
print("조용한 / 덜컹거리는 재질별 AUROC")
display(g[[(m, k) for m in MODELS for k in ("quiet", "loud")]].round(3))

In [ ]:
print("빠진 재질별 AUROC — proto(제안 방법)")
display(df[df.method == "proto"].pivot_table(index="heldout", columns="model", values="auroc").loc[order, MODELS].round(3))

best = model_table("auroc").stack().sort_values(ascending=False).head(5)
print("AUROC 상위 5개 (방법, 모델):")
display(best.round(3))

## 4. 원인 진단 비교 (D-02와 같은 지표)

- `auroc_train_known`: 처음 보는 녹화가 학습 녹화보다 먼가 (0.5면 녹화 차이 없음)
- `auroc_train_unknown`: 본 데이터와 새 재질을 구분하는가
- `known_acc`: 처음 보는 녹화의 알려진 재질을 가장 가까운 프로토타입으로 맞힌 비율

In [ ]:
cols = ["auroc_train_known", "auroc_train_unknown", "auroc_known_unknown", "known_acc",
        "med_train", "med_known", "med_unknown"]
display(diag.groupby("model")[cols].mean().loc[MODELS].round(3))

## 5. τ 판정

- `known_pass`: 알려진 재질 테스트 구간 통과율 (목표 0.95)
- `unknown_reject`: 빠진 재질 구간을 미등록으로 판정한 비율

In [ ]:
tau = df[df.method.isin(TAU_METHODS)].pivot_table(index="method", columns="model",
                                                    values=["known_pass", "unknown_reject"])
display(tau.loc[list(TAU_METHODS)].round(3))
print("proto, 빠진 재질별 미등록 거부율")
display(df[df.method == "proto"].pivot_table(index="heldout", columns="model", values="unknown_reject")
        .loc[order, MODELS].round(3))

## 6. 그림

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
model_table("auroc").plot.bar(ax=axes[0], width=0.8)
axes[0].set_title("mean AUROC (147 splits)"); axes[0].set_ylim(0, 1)
axes[0].axhline(0.5, color="gray", ls="--", lw=1)

p = df[df.method == "proto"].pivot_table(index="group", columns="model", values="auroc").loc[["quiet", "loud"], MODELS]
p.plot.bar(ax=axes[1], width=0.7)
axes[1].set_title("proto AUROC: quiet vs loud held-out"); axes[1].set_ylim(0, 1)
axes[1].axhline(0.5, color="gray", ls="--", lw=1)

model_table("rho_vib").plot.bar(ax=axes[2], width=0.8)
axes[2].set_title("Spearman(score, vibration)"); axes[2].axhline(0, color="black", lw=1)

for ax in axes:
    ax.tick_params(axis="x", rotation=30); ax.set_xlabel("")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False)
plt.tight_layout()
plt.savefig(OUT / "D_compare_models.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. results.csv 기록 (fcn, fcn-center만)

mlp는 D-01·D-03에서 이미 기록함. 방법 × 빠진 재질별 fold 평균

In [ ]:
new = df[df.model != "mlp"]
rows = new.groupby(["model", "method", "heldout"])[["auroc", "fpr95"]].mean().reset_index()
for _, r in rows.iterrows():
    log_result("D", f"{r.model}+{r.method}", {"auroc": r.auroc, "fpr95": r.fpr95},
               setting=f"heldout={r.heldout}", fold="mean",
               note=f"{DIFFICULTY.get(r.heldout, 'mid')}, D-04")

import shutil
shutil.copy(ROOT / "results" / "results.csv", OUT / "results_with_D04.csv")
print(len(rows), "줄 기록")

## 메모할 것

- center loss로 proto AUROC가 올랐는지 (mlp → fcn → fcn-center):
- 진동 상관이 줄었는지, 조용한 재질을 잡기 시작했는지:
- 처음 보는 녹화 분류 정확도(`known_acc`)가 달라졌는지:
- τ에서 알려진 재질 통과율 / 미등록 거부율: